In [40]:
2+2


4

In [41]:

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()


True

In [42]:
llm=ChatOpenAI(model="gpt-4o-mini",temperature=0.5)

In [43]:

def add_tool(a:int,b:int)->int:
    """This tool accepts two paramerters of type int for addition
    and returns the output as sum of two numbers in int"""
    return a+b

In [44]:

def add_multiply(a:int,b:int)->int:
    """This tool accepts two paramerters of type int for multiply
    and returns the output as product of two numbers in int"""
    return a*b

In [45]:
tools=[add_tool,add_multiply]
llm_with_tools=llm.bind_tools(tools)

In [46]:
llm_with_tools.invoke(" what is the sum of 4 and 5")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 122, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CvcygrueAQghibPjGL1QKGtAqcrG2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b9c1a-d06f-72a2-9e9e-110669581df5-0', tool_calls=[{'name': 'add_tool', 'args': {'a': 4, 'b': 5}, 'id': 'call_UbBbGhuKjBrD2NO7OsfvoBWy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens': 18, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [47]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.checkpoint.memory import InMemorySaver


In [48]:
class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

In [49]:
def llm_calling(state:State):
    return {
        "messages":[llm_with_tools.invoke(state["messages"])]
    }

In [50]:
graph=StateGraph(State)

graph.add_node("llm_calling",llm_calling)
graph.add_node("tools",ToolNode(tools=tools,messages_key="messages"))

graph.add_edge(START,"llm_calling")
graph.add_conditional_edges("llm_calling",tools_condition)
graph.add_edge("tools","llm_calling")
graph.add_edge("llm_calling",END)

memory=InMemorySaver()
workflow=graph.compile(checkpointer=memory)

config={"configurable":{"thread_id":1}}

In [51]:
print(workflow.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm_calling(llm_calling)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm_calling;
	llm_calling -.-> __end__;
	llm_calling -.-> tools;
	tools --> llm_calling;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [53]:
response=workflow.invoke({"messages":"what is the sum of 3 and 5"},config=config,stream_mode="values")
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

sum of 3+3
================================== Ai Message ==================================
Tool Calls:
  add_tool (call_ma1jBb0j2HmLV4R68QajjpRl)
 Call ID: call_ma1jBb0j2HmLV4R68QajjpRl
  Args:
    a: 3
    b: 3
================================= Tool Message =================================
Name: add_tool

6
================================== Ai Message ==================================

The sum of 3 + 3 is 6.
================================ Human Message =================================

what is the sum of 3 and 5
================================== Ai Message ==================================
Tool Calls:
  add_tool (call_hRm1GaY0icxEti68X7ArSeis)
 Call ID: call_hRm1GaY0icxEti68X7ArSeis
  Args:
    a: 3
    b: 5
================================= Tool Message =================================
Name: add_tool

8
================================== Ai Message ==================================

The sum o